# PROJECT: MietRecht-AI-Assistant/

# 2. `src/analyzer.py`

* Goal: Handle the PDF "heavy lifting" and clause detection.

In [ ]:
import re
from llama_index.readers.file import PDFReader
from llama_index.core import Settings


def detect_clause_type(text):
    keywords = {
        "Kündigung (Termination)": ["kündigung", "frist", "termination", "notice"],
        "Schönheitsreparaturen (Repairs)": ["renovier", "maler", "repairs", "schönheit"],
        "Kaution (Deposit)": ["kaution", "sicherheit", "deposit"],
        "Nebenkosten (Utilities)": ["betriebskosten", "utility", "nebenkosten"]
    }
    for label, keys in keywords.items():
        if any(k in text.lower() for k in keys): return label
    return "Allgemeine Klausel / General Clause"

def process_lease(file_obj):
    if file_obj is None: return "Please upload a PDF."
    try:
        loader = PDFReader()
        documents = loader.load_data(file=file_obj.name)
        full_text = "\n".join([d.text for d in documents])

        sections = re.split(r'(\n§\s*\d+)', full_text)
        final_output = "# 📑 Section-by-Section Analysis\n\n"

        count = 0
        for i in range(1, len(sections), 2):
            if count >= 8: break
            clause_text = sections[i] + (sections[i+1] if i+1 < len(sections) else "")
            c_type = detect_clause_type(clause_text)

            prompt = f"System: You are a German legal expert. Analyze this clause. 1. Summarize in EN/DE. 2. Give a risk score 0-10. Clause: {clause_text}"
            response = Settings.llm.complete(prompt)
            final_output += f"### {c_type}\n{response.text}\n\n---\n"
            count += 1
        return final_output
    except Exception as e:
        return f"Error: {str(e)}"